## Import Libs

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Import for get the environment variables 
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

MINIO_CONTAINER=os.getenv("MINIO_CONTAINER")
MINIO_USER=os.getenv("MINIO_USER")
MINIO_PASSWORD=os.getenv("MINIO_PASSWORD")

## Configs Spark

In [2]:
conf = SparkConf()

conf.setAppName("Delta Table History") # Name of spark application, useful for logs
conf.set("spark.hadoop.fs.s3a.endpoint",f"http://{MINIO_CONTAINER}:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", f"{MINIO_USER}") # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", f"{MINIO_PASSWORD}") # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True)
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

## Add data and schema for Data Frame

In [4]:
data = [("Rodrigo", "Souza", "M", 3000),
       ("Rafael", "Silva", "M", 3500),
        ("Laura", "Alvez", "M", 3250)
       ]

schema = StructType([
    StructField("firstName", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("salary", IntegerType(), True)
])

df = spark.createDataFrame(data=data, schema=schema)
df.show()

+---------+--------+------+------+
|firstName|lastName|gender|salary|
+---------+--------+------+------+
|  Rodrigo|   Souza|     M|  3000|
|   Rafael|   Silva|     M|  3500|
|    Laura|   Alvez|     M|  3250|
+---------+--------+------+------+



## Persist the Data Frame

In [6]:
df.write.format("delta").mode("append").save("s3a://bronze/table_schema_evolution")

## Show the Schema on minIo S3

In [8]:
spark.read.format("delta").load("s3a://bronze/table_schema_evolution").show()

+---------+--------+------+------+
|firstName|lastName|gender|salary|
+---------+--------+------+------+
|  Rodrigo|   Souza|     M|  3000|
|   Rafael|   Silva|     M|  3500|
|    Laura|   Alvez|     M|  3250|
+---------+--------+------+------+



## New Schema

In [5]:
new_data = [("Roberto", "Toledo", "M", 4500, 30)]

new_schema = StructType([
    StructField("firstName", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("age", IntegerType(), True)
])

df_new = spark.createDataFrame(data=new_data, schema=new_schema)
df_new.show()

+---------+--------+------+------+---+
|firstName|lastName|gender|salary|age|
+---------+--------+------+------+---+
|  Roberto|  Toledo|     M|  4500| 30|
+---------+--------+------+------+---+



## Write a new schema on table already exists

In [11]:
df_new.write.format("delta").option("mergeSchema", "true").mode("append").save("s3a://bronze/table_schema_evolution")

## Show the new table with the new schema

In [12]:
spark.read.format("delta").load("s3a://bronze/table_schema_evolution").show()

+---------+--------+------+------+----+
|firstName|lastName|gender|salary| age|
+---------+--------+------+------+----+
|  Roberto|  Toledo|     M|  4500|  30|
|  Rodrigo|   Souza|     M|  3000|null|
|   Rafael|   Silva|     M|  3500|null|
|    Laura|   Alvez|     M|  3250|null|
+---------+--------+------+------+----+

